# SupplySight AI — Data Understanding

**Phase:** Data Understanding / Exploratory Analysis  
**Objective:** Profile raw DataCo Supply Chain datasets for enterprise analytics readiness.

> **Guardrails:** This notebook does **not** clean data, rename columns, build dashboards, or train models. Analysis is read-only against raw CSVs.


## 1. Environment & Libraries

In [ ]:
# Core analysis stack for enterprise data understanding
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import missingno as msno

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

ROOT = Path("..") if Path("..", "data").exists() else Path(".")
# Prefer project-relative paths when notebook runs from analysis/
if not (ROOT / "data" / "raw" / "data" / "DataCoSupplyChainDataset.csv").exists():
    ROOT = Path(r"D:/Data Science Project/SupplySightAi")

RAW_MAIN = ROOT / "data" / "raw" / "data" / "DataCoSupplyChainDataset.csv"
RAW_DESC = ROOT / "data" / "raw" / "data" / "DescriptionDataCoSupplyChain.csv"
OUTPUT = Path("output") if Path("output").exists() or Path(".").resolve().name == "analysis" else ROOT / "analysis" / "output"
if not str(OUTPUT).endswith("output"):
    OUTPUT = ROOT / "analysis" / "output"
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / "figures").mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT.resolve())
print("OUTPUT:", OUTPUT.resolve())


## 2. Load Raw Datasets

Load both CSVs with `latin-1` encoding (DataCo standard). **No transformations.**

In [ ]:
# Load transactional supply-chain dataset and field dictionary
df = pd.read_csv(RAW_MAIN, encoding="latin-1", low_memory=False)
desc = pd.read_csv(RAW_DESC, encoding="latin-1")

print("Main dataset loaded:", df.shape)
print("Description dictionary loaded:", desc.shape)
desc.head(10)


## 3. Dataset Overview

In [ ]:
# High-level shape, preview, schema, and memory footprint
print("Dataset Shape:", df.shape)
print("Number of Rows:", len(df))
print("Number of Columns:", df.shape[1])
print("\nMemory Usage (deep): {:.2f} MB".format(df.memory_usage(deep=True).sum() / 1024**2))

print("\n--- First 5 Rows ---")
display(df.head())

print("--- Last 5 Rows ---")
display(df.tail())

print("--- Column Names ---")
print(list(df.columns))

print("\n--- Data Types ---")
display(df.dtypes.to_frame("dtype"))

print("--- Memory Usage by Column (top 15) ---")
display(df.memory_usage(deep=True).sort_values(ascending=False).head(15).to_frame("bytes"))


## 4. Column Analysis

Build a column dictionary profile and persist to `output/column_summary.csv`.

In [ ]:
# Column-level profile: dtype, missingness, cardinality, example value
rows = []
for col in df.columns:
    s = df[col]
    non_null = s.dropna()
    missing = int(s.isna().sum())
    rows.append({
        "Column Name": col,
        "Data Type": str(s.dtype),
        "Missing Values": missing,
        "Missing Percentage": round(100.0 * missing / len(df), 4),
        "Unique Values": int(s.nunique(dropna=True)),
        "Example Value": non_null.iloc[0] if len(non_null) else np.nan,
    })

column_summary = pd.DataFrame(rows)
column_summary.to_csv(OUTPUT / "column_summary.csv", index=False)
display(column_summary)


## 5. Missing Value Analysis

In [ ]:
# Missing value table + missingno visuals
missing_report = (
    pd.DataFrame({
        "Column": df.columns,
        "Missing Values": df.isna().sum().values,
        "Missing Percentage": np.round(100.0 * df.isna().sum().values / len(df), 4),
    })
    .sort_values("Missing Values", ascending=False)
    .reset_index(drop=True)
)
missing_report.to_csv(OUTPUT / "missing_values.csv", index=False)
display(missing_report[missing_report["Missing Values"] > 0])

miss_cols = df.columns[df.isna().any()].tolist()
sample = df if len(df) <= 5000 else df.sample(5000, random_state=42)
plot_df = sample[miss_cols] if miss_cols else sample.iloc[:, :20]

msno.matrix(plot_df, figsize=(14, 6), fontsize=9)
plt.title("Missing Value Matrix")
plt.show()

msno.bar(plot_df, figsize=(14, 6), fontsize=9)
plt.title("Missing Value Bars")
plt.show()


## 6. Duplicate Analysis

In [ ]:
# Full-row duplicate assessment (no rows dropped)
dup_count = int(df.duplicated().sum())
duplicate_report = pd.DataFrame([{
    "Total Rows": len(df),
    "Duplicate Rows": dup_count,
    "Duplicate Percentage": round(100.0 * dup_count / len(df), 6),
    "Unique Rows": len(df) - dup_count,
}])
duplicate_report.to_csv(OUTPUT / "duplicate_report.csv", index=False)
display(duplicate_report)


## 7. Numerical Analysis

Summary statistics, distributions, boxplots, and IQR outlier profiling.

In [ ]:
# Numeric describe + plots + outlier table
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
statistics = df[numeric_cols].describe().T
statistics.to_csv(OUTPUT / "statistics.csv")
display(statistics)

# Prefer business metrics for readable plots
preferred = [
    "Days for shipping (real)", "Days for shipment (scheduled)", "Benefit per order",
    "Sales per customer", "Late_delivery_risk", "Order Item Discount",
    "Order Item Discount Rate", "Order Item Product Price", "Order Item Profit Ratio",
    "Order Item Quantity", "Sales", "Order Item Total", "Order Profit Per Order",
    "Product Price", "Product Status",
]
plot_cols = [c for c in preferred if c in df.columns]

fig, axes = plt.subplots(int(np.ceil(len(plot_cols)/3)), 3, figsize=(14, 3.2 * np.ceil(len(plot_cols)/3)))
axes = np.array(axes).reshape(-1)
for i, col in enumerate(plot_cols):
    axes[i].hist(df[col].dropna(), bins=40, color="#1f4e79", alpha=0.85)
    axes[i].set_title(col, fontsize=9)
for j in range(i+1, len(axes)):
    axes[j].axis("off")
fig.suptitle("Histograms — Key Numerical Features")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(int(np.ceil(len(plot_cols)/3)), 3, figsize=(14, 3.2 * np.ceil(len(plot_cols)/3)))
axes = np.array(axes).reshape(-1)
for i, col in enumerate(plot_cols):
    axes[i].boxplot(df[col].dropna(), vert=True)
    axes[i].set_title(col, fontsize=9)
for j in range(i+1, len(axes)):
    axes[j].axis("off")
fig.suptitle("Boxplots — Outlier View")
plt.tight_layout()
plt.show()

outlier_rows = []
for col in numeric_cols:
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = int(((s < lower) | (s > upper)).sum())
    outlier_rows.append({
        "Column": col, "Outlier Count": outliers,
        "Outlier Percentage": round(100.0 * outliers / len(s), 4),
        "Lower Bound": lower, "Upper Bound": upper,
    })
outlier_df = pd.DataFrame(outlier_rows).sort_values("Outlier Percentage", ascending=False)
display(outlier_df.head(15))


## 8. Categorical Analysis

In [ ]:
# Categorical / object column frequency profiles
cat_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
# Include low-cardinality integer flags often used as categoricals
for col in ["Late_delivery_risk", "Product Status"]:
    if col in df.columns and col not in cat_cols:
        cat_cols.append(col)

cat_rows = []
for col in cat_cols:
    vc = df[col].value_counts(dropna=False).head(10)
    top_str = "; ".join([f"{idx} ({cnt})" for idx, cnt in vc.items()])
    cat_rows.append({
        "Column": col,
        "Unique Values": int(df[col].nunique(dropna=True)),
        "Top Categories (value count)": top_str,
        "Most Frequent": vc.index[0] if len(vc) else np.nan,
        "Most Frequent Count": int(vc.iloc[0]) if len(vc) else 0,
    })

categorical_summary = pd.DataFrame(cat_rows)
categorical_summary.to_csv(OUTPUT / "categorical_summary.csv", index=False)
display(categorical_summary)

# Interactive Plotly bar for a key business categorical
if "Delivery Status" in df.columns:
    status_counts = df["Delivery Status"].value_counts().reset_index()
    status_counts.columns = ["Delivery Status", "Count"]
    fig = px.bar(status_counts, x="Delivery Status", y="Count", title="Delivery Status Frequency")
    fig.show()


## 9. Date Analysis

Automatically detect date-like columns and summarize timeline coverage.

In [ ]:
# Detect and profile date/datetime columns without mutating source df permanently
date_candidates = [c for c in df.columns if any(t in c.lower() for t in ("date", "time", "timestamp"))]
for col in df.select_dtypes(include="object").columns:
    if col in date_candidates:
        continue
    sample = df[col].dropna().astype(str).head(50)
    if sample.empty:
        continue
    parsed = pd.to_datetime(sample, errors="coerce")
    if parsed.notna().mean() >= 0.8:
        date_candidates.append(col)

print("Detected date columns:", date_candidates)
date_info = {}
for col in date_candidates:
    parsed = pd.to_datetime(df[col], errors="coerce")
    valid = parsed.dropna()
    if valid.empty:
        continue
    date_info[col] = {
        "min": valid.min(),
        "max": valid.max(),
        "span_days": (valid.max() - valid.min()).days,
        "non_null": int(valid.shape[0]),
        "nulls": int(parsed.isna().sum()),
    }
    print(f"\n{col}: min={valid.min()} | max={valid.max()} | span_days={(valid.max()-valid.min()).days}")

# Timeline of record volume for primary order date if available
primary = "order date (DateOrders)" if "order date (DateOrders)" in df.columns else (date_candidates[0] if date_candidates else None)
if primary:
    parsed = pd.to_datetime(df[primary], errors="coerce")
    daily = parsed.dt.to_period("M").value_counts().sort_index()
    daily.index = daily.index.astype(str)
    fig = px.line(x=daily.index, y=daily.values, labels={"x": "Month", "y": "Records"}, title=f"Record Timeline — {primary}")
    fig.show()


## 10. Correlation Analysis

In [ ]:
# Correlation matrix among business numeric measures (IDs excluded from heatmap focus)
exclude = {
    "customer id", "order id", "order item id", "order customer id", "product card id",
    "order item cardprod id", "category id", "department id", "product category id",
    "customer zipcode", "order zipcode",
}
corr_cols = [c for c in numeric_cols if c.lower() not in exclude]
corr = df[corr_cols].corr(numeric_only=True)
display(corr.round(3))

plt.figure(figsize=(14, 11))
sns.heatmap(corr, cmap="RdBu_r", center=0, square=True, linewidths=0.2, cbar_kws={"shrink": 0.7})
plt.title("Correlation Heatmap — Numerical Features")
plt.tight_layout()
plt.show()

# Highlight strongest absolute correlations
pairs = []
cols = corr.columns.tolist()
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        val = corr.iloc[i, j]
        if pd.notna(val):
            pairs.append((cols[i], cols[j], float(val)))
pairs.sort(key=lambda x: abs(x[2]), reverse=True)
print("Top absolute correlations:")
for a, b, v in pairs[:15]:
    print(f"  {a}  <->  {b}: {v:.4f}")


## 11. Business Understanding

### Orders
The dataset is primarily at **order-item grain** (`Order Id` + `Order Item Id`). Use this carefully when computing order-level KPIs.

### Customers
Customer identity and segmentation fields (`Customer Id`, `Customer Segment`, geography, contact fields) support cohort and segment analysis. Email/password are PII.

### Products & Categories
Product catalog attributes (`Product Name`, `Product Price`, `Category Name`, `Department Name`) enable assortment and pricing analytics.

### Shipping / Logistics
`Shipping Mode`, scheduled vs real shipping days, `Delivery Status`, and `Late_delivery_risk` are central to SLA and delay analytics.

### Sales & Profit
`Sales`, discounts, totals, benefit/profit fields support revenue and margin monitoring.

### Geography
Market, region, country, city, and coordinates enable regional performance and logistics views.

### Inventory
Explicit on-hand inventory is limited; future inventory optimization may require warehouse stock feeds beyond this extract.


## 12. Export Confirmation

All tabular artifacts written under `analysis/output/`. Full narrative report: `analysis/Data_Understanding_Report.md`.

In [ ]:
# Confirm exports exist
for name in [
    "column_summary.csv",
    "missing_values.csv",
    "duplicate_report.csv",
    "statistics.csv",
    "categorical_summary.csv",
]:
    path = OUTPUT / name
    print(("[OK]" if path.exists() else "[MISSING]"), path)
